# Question 8  
In this exercise, we will generate simulated data, and will then use this data to perform forward and backward stepwise selection.

### (a)    
Create a random number generator and use its normal() method to generate a predictor $X$ of length $n = 100$, as well as a noise vector $\epsilon$ of length $n = 100$.

In [1]:
import numpy as np
n = 100
X = np.random.normal(0,1,n)
epsilon = np.random.normal(0,1,n) 

### (b)  
Generate a response vector $Y$ of length $n = 100$ according to the model$$Y = \beta_0 + \beta_1 X + \beta_2 X^2 + \beta_3 X^3 + \epsilon,$$where $\beta_0$, $\beta_1$, $\beta_2$, and $\beta_3$ are constants of your choice.

In [2]:
beta0 = 1
beta1 = 1
beta2 = 1
beta3 = 1

Y = beta0 + beta1*X + beta2*(X**2) + beta3*(X**3) + epsilon

### (c)  
Use forward stepwise selection in order to select a model containing the predictors $X, X^2, \dots, X^{10}$. What is the model obtained according to $C_p$? $$C_p = \frac{RSS_d}{\hat{\sigma}^2} - n + 2d$$ Report the coefficients of the model obtained.

In [4]:
import pandas as pd

data = pd.DataFrame({'Y':Y})
for i in range(1,11):
    data[f'X{i}'] = X**i
predictors = [f'X{i}' for i in range(1,11)]
print(data.head(5))


           Y        X1        X2        X3         X4         X5         X6  \
0   2.281805  0.573662  0.329088  0.188785   0.108299   0.062127   0.035640   
1   6.703123  1.067999  1.140622  1.218183   1.301018   1.389486   1.483969   
2   1.197844  0.156385  0.024456  0.003825   0.000598   0.000094   0.000015   
3  11.608601  1.911178  3.652600  6.980769  13.341490  25.497958  48.731131   
4  -1.331721 -0.659353  0.434746 -0.286651   0.189004  -0.124621   0.082169   

          X7            X8            X9           X10  
0   0.020445  1.172869e-02  6.728304e-03  3.859773e-03  
1   1.584877  1.692647e+00  1.807745e+00  1.930670e+00  
2   0.000002  3.577417e-07  5.594559e-08  8.749073e-09  
3  93.133853  1.779953e+02  3.401808e+02  6.501459e+02  
4  -0.054178  3.572262e-02 -2.355381e-02  1.553028e-02  


In [19]:
import statsmodels.api as sm

def get_rss(y, X_data):
    model = sm.OLS(y, sm.add_constant(X_data)).fit()
    return model.ssr

def forward_selection(data, predictors, response_col='Y'):
    remaining_predictors = set(predictors)
    selected_predictors = []
    models_fit = [] 
    
    
    for i in range(1, len(predictors) + 1):
        best_rss = np.inf
        best_feature = None
        
        for feature in remaining_predictors:
            current_predictors = selected_predictors + [feature]
            rss = get_rss(data[response_col], data[current_predictors])
            
            if rss < best_rss:
                best_rss = rss
                best_feature = feature
        
        selected_predictors.append(best_feature)
        remaining_predictors.remove(best_feature)
        models_fit.append({'d': i, 'rss': best_rss, 'predictors': selected_predictors.copy()})
        
    return pd.DataFrame(models_fit)


forward_results = forward_selection(data, predictors)

full_model = sm.OLS(data['Y'], sm.add_constant(data[predictors])).fit()
sigma_hat_sq = full_model.ssr / (n - 11)

forward_results['Cp'] = (forward_results['rss'] / sigma_hat_sq) - n + 2 * forward_results['d']

best_forward_idx = forward_results['Cp'].idxmin()
best_forward_model_info = forward_results.loc[best_forward_idx]

print("\033[1mANS for question C:\033[0m")
print(f" {int(best_forward_model_info['d'])} variables in the best model based on the value of Cp")
print(f"selected variables: {best_forward_model_info['predictors']}")
print(f"minimum value of Cp : {best_forward_model_info['Cp']:.4f}")

final_model = sm.OLS(data['Y'], sm.add_constant(data[best_forward_model_info['predictors']])).fit()
print("coefficient：")
print(final_model.params)

ANS for question C:
 3 variables in the best model based on the value of Cp
selected variables: ['X3', 'X2', 'X1']
minimum value of Cp : 2.8318
coefficient：
const    1.010633
X3       0.953256
X2       0.952593
X1       1.231519
dtype: float64


### (d)  
Repeat (c), using backwards stepwise selection. How does your answer compare to the results in (c)?

### (e)  
Now fit a lasso model to the simulated data, again using $X, X^2, \dots, X^{10}$ as predictors. Use cross-validation to select the optimal value of $\lambda$. Create plots of the cross-validation error as a function of $\lambda$. Report the resulting coefficient estimates, and discuss the results obtained.

### (f)  
Now generate a response vector $Y$ according to the model$$Y = \beta_0 + \beta_7 X^7 + \epsilon,$$and perform forward stepwise selection and the lasso. Discuss the results obtained.